# Rock — matched pivot into analysis workbook

Builds a pivot from `Rock_royalties_2021Q1_2026Q1_2_matched.csv` and writes it to the **data** tab of `Rock_royalties_2021Q1_2026Q1_4_matched_analysis.xlsx` in `_output`. Existing data on that tab is replaced; other tabs are kept. If the workbook does not exist, it is created.

Use **Run All**. A timestamped run log is written to the same folder.


In [5]:

# this script creates a pivot table from the matched Rock statements

import os
import sys
import zipfile
import tempfile
import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
from xml.sax.saxutils import escape

import pandas as pd

karen_root = '/Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM group/Royalties/Statements/Karen'

file = f'{karen_root}/Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_2_matched.csv'
analysisfile_name = 'Rock_royalties_2021Q1_2026Q1_3_matched_analysis.xlsx'
consolidated_dir = f'{karen_root}/Rock Music/Consolidated statements/'
outputdirectory = f'{karen_root}/_output/'
sheet_name = 'data'


def resolve_dir(rel):
    cwd = os.getcwd()
    alt = rel.replace('../../', '../', 1) if rel.startswith('../../') else rel
    candidates = [
        os.path.abspath(os.path.join(cwd, rel)),
        os.path.abspath(os.path.join(cwd, 'Karen_statements', rel)),
        os.path.abspath(os.path.join(cwd, alt)),
        os.path.abspath(os.path.join(os.path.dirname(cwd), rel)),
        os.path.abspath(os.path.join(cwd, '..', alt)),
    ]
    for p in candidates:
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(f'Directory not found: {rel}\nTried:\n- ' + '\n- '.join(candidates))


consolidated_dir = resolve_dir(consolidated_dir)
outputdirectory = resolve_dir(outputdirectory)
output_path = os.path.join(outputdirectory, analysisfile_name)
analysis_candidates = [
    os.path.join(outputdirectory, analysisfile_name),
    os.path.join(consolidated_dir, analysisfile_name),
]
analysis_path = next((p for p in analysis_candidates if os.path.isfile(p)), None)

os.makedirs(outputdirectory, exist_ok=True)
logfile_name = (
    os.path.splitext(analysisfile_name)[0]
    + '_run_log_'
    + datetime.now().strftime('%Y%m%d_%H%M%S')
    + '.txt'
)
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout


class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()

    def flush(self):
        for s in self.streams:
            s.flush()

    def isatty(self):
        return False

    def __getattr__(self, name):
        return getattr(self.streams[0], name)


sys.stdout = _Tee(_original_stdout, _log_file)
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 20)


def close_log():
    sys.stdout.flush()
    sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f'Run log saved to: {log_path}')


def fmt_int(n):
    try:
        return f'{int(n):,}'
    except (TypeError, ValueError):
        return str(n)


def fmt_money(n):
    try:
        return f'{float(n):,.2f}'
    except (TypeError, ValueError):
        return str(n)


def fmt_units(n):
    try:
        return f'{float(n):,.2f}'
    except (TypeError, ValueError):
        return str(n)


def fmt_shape(df):
    return f'{fmt_int(df.shape[0])} rows × {len(df.columns)} columns'


def header(title):
    line = '=' * 72
    print(f'\n{line}\n  {title}\n{line}')


def subheader(title):
    print(f'\n--- {title} ---')


def print_totals(label, df, royalty_col='AMOUNT (HKD)', units_col='UNIT'):
    royalty = df[royalty_col].sum()
    units = df[units_col].sum()
    print(
        f'  {label:<24} AMOUNT (HKD): {fmt_money(royalty):>16}'
        f'    UNIT: {fmt_units(units):>20}    ({fmt_shape(df)})'
    )


def _col_letter(n):
    letters = ''
    while n:
        n, remainder = divmod(n - 1, 26)
        letters = chr(65 + remainder) + letters
    return letters


def _xml_text(value):
    text = ''.join(ch for ch in str(value) if ch in '\t\n\r' or ord(ch) >= 32)
    return escape(text)


def _write_sheet_xml(df, xml_path):
    """Write a worksheet using inline strings so the rest of the xlsx can keep its sharedStrings.xml."""
    n_rows, n_cols = df.shape
    last_cell = f'{_col_letter(n_cols)}{n_rows + 1}'
    numeric_cols = [pd.api.types.is_numeric_dtype(df[col]) for col in df.columns]
    with open(xml_path, 'w', encoding='utf-8') as fh:
        fh.write('<?xml version="1.0" encoding="UTF-8" standalone="yes"?>')
        fh.write('<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main">')
        fh.write(f'<dimension ref="A1:{last_cell}"/>')
        fh.write('<sheetData>')
        fh.write('<row r="1">')
        for idx, col in enumerate(df.columns, start=1):
            fh.write(
                f'<c r="{_col_letter(idx)}1" t="inlineStr"><is><t>{_xml_text(col)}</t></is></c>'
            )
        fh.write('</row>')
        values = df.to_numpy()
        for row_idx, row in enumerate(values, start=2):
            fh.write(f'<row r="{row_idx}">')
            for col_idx, value in enumerate(row):
                if value is None or (isinstance(value, float) and pd.isna(value)) or pd.isna(value):
                    continue
                cell_ref = f'{_col_letter(col_idx + 1)}{row_idx}'
                if numeric_cols[col_idx]:
                    fh.write(f'<c r="{cell_ref}" t="n"><v>{value}</v></c>')
                else:
                    fh.write(
                        f'<c r="{cell_ref}" t="inlineStr"><is><t xml:space="preserve">{_xml_text(value)}</t></is></c>'
                    )
            fh.write('</row>')
        fh.write('</sheetData></worksheet>')


def _sheet_zip_path(xlsx_path, name):
    ns = {
        'm': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main',
        'r': 'http://schemas.openxmlformats.org/officeDocument/2006/relationships',
    }
    with zipfile.ZipFile(xlsx_path) as zf:
        workbook = ET.fromstring(zf.read('xl/workbook.xml'))
        rels = ET.fromstring(zf.read('xl/_rels/workbook.xml.rels'))
    rel_id = None
    for sheet in workbook.findall('m:sheets/m:sheet', ns):
        if sheet.get('name') == name:
            rel_id = sheet.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
            break
    if rel_id is None:
        return None
    target = None
    for rel in rels:
        if rel.get('Id') == rel_id:
            target = rel.get('Target')
            break
    if not target:
        return None
    target = target.lstrip('/')
    if target.startswith('xl/'):
        return target
    return 'xl/' + target


def save_pivot_into_analysis(pivot_df, source_path, dest_path, name='data'):
    """Write the pivot to tab `name`. Keep other sheets if the workbook already exists."""
    if source_path and os.path.isfile(source_path):
        sheet_part = _sheet_zip_path(source_path, name)
        if sheet_part is not None:
            print(f'  Loading analysis file : {source_path}')
            print(f'  Clearing tab          : {name} ({sheet_part})')
            with tempfile.TemporaryDirectory() as tmpdir:
                xml_path = os.path.join(tmpdir, 'sheet.xml')
                _write_sheet_xml(pivot_df, xml_path)
                sheet_bytes = Path(xml_path).read_bytes()
                tmp_out = dest_path + '.tmp'
                with zipfile.ZipFile(source_path, 'r') as zin, zipfile.ZipFile(tmp_out, 'w') as zout:
                    for item in zin.infolist():
                        if item.filename == sheet_part:
                            zi = zipfile.ZipInfo(item.filename, item.date_time)
                            zi.compress_type = zipfile.ZIP_DEFLATED
                            zout.writestr(zi, sheet_bytes)
                        else:
                            zout.writestr(item, zin.read(item.filename))
                os.replace(tmp_out, dest_path)
            return
        print(f'  Workbook found but no "{name}" tab — adding it')
        from openpyxl import load_workbook
        import shutil
        if os.path.abspath(source_path) != os.path.abspath(dest_path):
            shutil.copy2(source_path, dest_path)
        with pd.ExcelWriter(dest_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            pivot_df.to_excel(writer, sheet_name=name, index=False)
        return

    print('  Analysis file not found — creating a new workbook')
    pivot_df.to_excel(dest_path, sheet_name=name, index=False, engine='openpyxl')


header('Rock — matched pivot')
print(f'  Run started              : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Input file               : {file}')
print(f'  Analysis file            : {analysisfile_name}')
print(f'  Output dir               : {outputdirectory}')
print(f'  Output file              : {output_path}')
print(f'  Sheet                    : {sheet_name}')
print(f'  Run log                  : {logfile_name}')

header('1. Required files')
if os.path.isfile(file):
    print(f'  [OK]       {file}')
else:
    print(f'  [MISSING]  {file}')
    close_log()
    raise FileNotFoundError(f'Stopping: required file not found:\n- {file}')

if analysis_path:
    print(f'  [OK]       {analysis_path}')
else:
    print(f'  [NEW]      {output_path}  (not found — a new workbook will be created)')

header('2. Load matched statements')
df = pd.read_csv(file, low_memory=False)
print(f'  Loaded                   : {fmt_shape(df)}')
print_totals('Input', df)

unnamed = [c for c in df.columns if str(c).startswith('Unnamed')]
if unnamed:
    print(f'  Dropping unnamed columns : {unnamed}')
    df = df.drop(columns=unnamed)

print()
print(f'  Columns ({len(df.columns)}):')
print('    ' + ', '.join(df.columns.astype(str)))

# Filter
df_merged = df   # no filter
# df_merged = df[df['USER'] == '滾石(北京)文化傳播有限公司'].iloc[:, :]
print_totals('Filtered', df_merged)



  Rock — matched pivot
  Run started              : 2026-08-19 17:36:34
  Input file               : ../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_2_matched.csv
  Analysis file            : Rock_royalties_2021Q1_2026Q1_3_matched_analysis.xlsx
  Output dir               : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM Group/Royalties/Statements/Karen/_output
  Output file              : /Users/johannesnatterer/Library/CloudStorage/OneDrive-NattererLabs/50 KM Group/Royalties/Statements/Karen/_output/Rock_royalties_2021Q1_2026Q1_3_matched_analysis.xlsx
  Sheet                    : data
  Run log                  : Rock_royalties_2021Q1_2026Q1_3_matched_analysis_run_log_20260819_173634.txt

  1. Required files
  [OK]       ../../50 KM Group/Royalties/Statements/Karen/Rock Music/Consolidated statements/Rock_royalties_2021Q1_2026Q1_2_matched.csv
  [OK]       /Users/johannesnatterer/Library/CloudStorage/

In [6]:

header('3. Build pivot')

val1 = 'UNIT'
val2 = 'AMOUNT (HKD)'
val3 = 'Amount to Rock (HKD)'

val1crit = 'sum'
val2crit = 'sum'
val3crit = 'sum'

x1 = 'Song'
x2 = 'Album'
x3 = 'Album Type'
x4 = 'Report Quarter'
x5 = 'USER'
x6 = 'Rev Quarter'
x7 = 'PayType'
x8 = 'Report year'

y1 = ''

sortby = val2

group_cols = [x8, x4, x7, x5, x3, x2, x1]
print(f'  Values                   : {val1} ({val1crit}), {val2} ({val2crit})')
print(f'  Index                    : {", ".join(group_cols)}')

missing_cols = [c for c in group_cols + [val1, val2] if c not in df_merged.columns]
if missing_cols:
    close_log()
    raise KeyError(f'Pivot is missing required columns: {missing_cols}')

for col in group_cols:
    empty = df_merged[col].isna().sum()
    if empty:
        print(f'  Filling {fmt_int(empty)} empty values in \'{col}\' with \'Blank\'')
    df_merged[col] = df_merged[col].fillna('Blank')

pivot = pd.pivot_table(
    df_merged,
    values=[val1, val2],
    index=group_cols,
    fill_value=0,
    margins=False,
    aggfunc='sum',
)
pivot = pivot.reset_index()
pivot[group_cols] = pivot[group_cols].ffill()

print(f'  Pivot shape              : {fmt_shape(pivot)}')
print_totals('Input', df_merged)
print_totals('Pivot', pivot)

subheader('Sample (first 10 rows)')
print(pivot.head(10).to_string(index=False))

header('4. Save output')
save_pivot_into_analysis(pivot, analysis_path, output_path, name=sheet_name)
print(f'  Wrote pivot to "{sheet_name}" : {fmt_shape(pivot)}')
print(f'  Saved                    : {output_path}')
print(f'  Run finished             : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
close_log()



  3. Build pivot
  Values                   : UNIT (sum), AMOUNT (HKD) (sum)
  Index                    : Report year, Report Quarter, PayType, USER, Album Type, Album, Song
  Pivot shape              : 41,578 rows × 9 columns
  Input                    AMOUNT (HKD):     3,951,607.08    UNIT:     7,354,374,169.00    (1,442,752 rows × 37 columns)
  Pivot                    AMOUNT (HKD):     3,951,607.08    UNIT:     7,354,374,169.00    (41,578 rows × 9 columns)

--- Sample (first 10 rows) ---
 Report year Report Quarter   PayType           USER Album Type           Album                 Song  AMOUNT (HKD)  UNIT
        2021             Q1 FullTrack            OSM      KM EP Come on Come on Come On Come On 光速飛翔          0.30     2
        2021             Q1 FullTrack            OSM      KM EP              再生             戀一世的愛(粵)          0.30     1
        2021             Q1 FullTrack            OSM   KM album       To Be 做自己                 他不愛我          1.58     7
        2021      